# Classic ML Baselines

This notebook runs the final classic baseline grid across multiple random seeds and creates analysis tables:

- `total_results`: all evaluated combinations across all seeds.
- `per_seed_report_results`: the best validation result per seed and representation family.
- `report_results`: mean/std validation metrics across seeds.

The task predicts sentiment labels `0..4` for mixed English/German product reviews. The validation score is `1 - MAE / 4`.

The final grid contains a majority baseline; sparse BoW/TF-IDF word/character features with Logistic Regression, Linear SVM, Ridge Classifier, and Complement NB; and static embedding features with average, TF-IDF weighted, and mean+max pooling trained with Logistic Regression, Linear SVM, and Ridge Classifier.


In [ ]:
from datetime import datetime
from pathlib import Path
import subprocess
import sys

import pandas as pd

EXPERIMENT_KIND = "CLASSIC_ML_BASELINES"
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
EXPERIMENT_DIR = Path("experiments/classic_ml") / f"{RUN_ID}_{EXPERIMENT_KIND}"
EMBEDDING_DIR = Path("experiments/embeddings")
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = Path("data/train.csv")
VALIDATION_SIZE = 0.1
SEEDS = [42, 43, 44, 45, 46]
MAX_FEATURES = 250_000
MAX_ITER = 100

embedding_paths = {
    "glove": EMBEDDING_DIR / "glove.6B.300d.txt",
    "fasttext": EMBEDDING_DIR / "fasttext.vec",
}

available_embeddings = {
    name: path for name, path in embedding_paths.items() if path.exists()
}
EXPERIMENT_DIR, SEEDS, available_embeddings


This notebook is configured for the final classic run: sparse lexical baselines plus dense static-embedding baselines for every seed. Place pretrained static embeddings in `experiments/embeddings/` with the filenames configured above.


In [ ]:
if not available_embeddings:
    raise FileNotFoundError("No embedding files found in experiments/embeddings/. Run load_embeddings.ipynb first.")

completed_runs = []
mode = "all"

for seed in SEEDS:
    seed_dir = EXPERIMENT_DIR / f"seed_{seed}"
    seed_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable,
        "-m",
        "baselines.classic_ml_baselines",
        "--mode",
        mode,
        "--train-path",
        str(TRAIN_PATH),
        "--output-dir",
        str(seed_dir),
        "--validation-size",
        str(VALIDATION_SIZE),
        "--random-state",
        str(seed),
        "--max-features",
        str(MAX_FEATURES),
        "--max-iter",
        str(MAX_ITER),
    ]
    for name, path in available_embeddings.items():
        cmd.extend([f"--{name}-path", str(path)])

    print(" ".join(cmd))
    subprocess.run(cmd, check=True)
    completed_runs.append({"seed": seed, "run_dir": seed_dir})

completed_runs


## Total Analysis

In [ ]:
result_frames = []
for run in completed_runs:
    frame = pd.read_csv(run["run_dir"] / "classic_ml_results.csv")
    frame.insert(0, "seed", run["seed"])
    result_frames.append(frame)

results = pd.concat(result_frames, ignore_index=True)
results.to_csv(EXPERIMENT_DIR / "classic_ml_results.csv", index=False)

total_results = results.sort_values(
    ["status", "cil_score"], ascending=[False, False]
).reset_index(drop=True)
total_results.to_csv(EXPERIMENT_DIR / "total_analysis.csv", index=False)
total_results


## Report Analysis

In [ ]:
ok_results = results[results["status"] == "ok"].copy()
per_seed_report_results = (
    ok_results.sort_values("cil_score", ascending=False)
    .groupby(["seed", "family"], as_index=False)
    .first()
    .sort_values(["family", "seed"])
    .reset_index(drop=True)
)
per_seed_report_results.to_csv(EXPERIMENT_DIR / "per_seed_report_analysis.csv", index=False)

report_results = (
    per_seed_report_results
    .groupby("family")[["cil_score", "mae", "accuracy", "macro_f1"]]
    .agg(["mean", "std"])
    .reset_index()
)
report_results.columns = [
    column[0] if column[1] == "" else f"{column[0]}_{column[1]}"
    for column in report_results.columns.to_flat_index()
]
report_results = report_results.sort_values("cil_score_mean", ascending=False).reset_index(drop=True)
report_results.to_csv(EXPERIMENT_DIR / "report_analysis.csv", index=False)
report_results


## Analysis Artifacts


In [ ]:
analysis_dir = EXPERIMENT_DIR / "analysis"
analysis_dir.mkdir(parents=True, exist_ok=True)

classic_fastest_table = analysis_dir / "classic_fastest_families.tex"
classic_full_table = analysis_dir / "classic_full_results.tex"
classic_seed_summary_table = analysis_dir / "classic_seed_summary.tex"
subprocess.run(
    [
        sys.executable,
        "-m",
        "baselines.analysis.make_classic_latex_table",
        "--input",
        str(EXPERIMENT_DIR / "classic_ml_results.csv"),
        "--output",
        str(classic_fastest_table),
        "--full-output",
        str(classic_full_table),
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "baselines.analysis.make_seed_summary_latex_table",
        "--input",
        str(EXPERIMENT_DIR / "per_seed_report_analysis.csv"),
        "--output",
        str(classic_seed_summary_table),
        "--group-by",
        "family",
        "--caption",
        "Classic baseline validation metrics across five seeds.",
        "--label",
        "tab:classic-seed-summary",
    ],
    check=True,
)

classic_fastest_table, classic_full_table, classic_seed_summary_table
